In [1]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14,6)


c:\Users\HA BUI\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# LOAD DATA
# ==============================

ohlc = pd.read_csv("data/ohlc.csv", parse_dates=["date"])
fundamental = pd.read_csv("data/fundamental.csv")
news = pd.read_csv("data/news.csv", parse_dates=["date"])

ohlc = ohlc.sort_values(["mack","date"])
fundamental = fundamental.sort_values(["mack","nam","quy"])
news = news.sort_values(["mack","date"])

print(ohlc.shape, fundamental.shape, news.shape)


(144176, 7) (2265, 11) (35481, 5)


In [ ]:
# TECHNICAL INDICATORS
# SMA, EMA
close = ohlc.groupby("mack")["close"]
vol = ohlc.groupby("mack")["volume"]

ohlc["sma_10"] = close.transform(lambda x: x.rolling(10).mean())
ohlc["sma_20"] = close.transform(lambda x: x.rolling(20).mean())
ohlc["sma_50"] = close.transform(lambda x: x.rolling(50).mean())
ohlc["ema_10"] = close.transform(lambda x: x.ewm(span=10, adjust=False).mean())
ohlc["ema_20"] = close.transform(lambda x: x.ewm(span=20, adjust=False).mean())

# RSI
def compute_rsi(x, window=14):
    delta = x.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    rs = gain.rolling(window).mean() / loss.rolling(window).mean()
    return 100 - (100 / (1 + rs))

ohlc["rsi_14"] = close.transform(compute_rsi)

ohlc["return"] = close.pct_change()

ema_12 = close.transform(lambda x: x.ewm(span=12, adjust=False).mean())
ema_26 = close.transform(lambda x: x.ewm(span=26, adjust=False).mean())
ohlc["macd"] = ema_12 - ema_26

ohlc["macd_signal"] = ohlc.groupby("mack")["macd"].transform(
    lambda x: x.ewm(span=9, adjust=False).mean()
)
ohlc["macd_hist"] = ohlc["macd"] - ohlc["macd_signal"]

ohlc["volatility_20"] = (
    ohlc.groupby("mack")["return"]
    .transform(lambda x: x.rolling(20).std())
)

vol_mean_20 = vol.transform(lambda x: x.rolling(20).mean()).replace(0, np.nan)
ohlc["volume_ratio"] = ohlc["volume"] / vol_mean_20

In [4]:
# FUNDAMENTAL → DAILY
fundamental["date"] = (
    pd.to_datetime(
        dict(year=fundamental["nam"], month=fundamental["quy"] * 3, day=1)
    ) + pd.offsets.MonthEnd(0)
)

fundamental_features = [
    "eps","roe","roa","pb","pe",
    "lnst_yoy","nophaitra_vcsh","vonhoa_tts"
]

fundamental_daily = (
    fundamental.sort_values(["mack", "date"])
    .set_index("date")
    .groupby("mack")[fundamental_features]
    .resample("D")
    .ffill()
    .reset_index()
)


In [5]:
#NEWS SENTIMENT – PHOBERT
# ==============================
# CUDA & PHOBERT
# ==============================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "wonrax/phobert-base-vietnamese-sentiment"
)

model = model.to(device)
model.eval()

print("✓ PhoBERT loaded successfully")



Using device: cpu
✓ PhoBERT loaded successfully


In [6]:

# PREPARE TEXT
# ==============================

news["text"] = news["title"].fillna("").astype(str)

news["text"] = news["text"].str.replace("\n", " ").str.strip()

news = news.reset_index(drop=True)

news[["date","mack","text"]].head()


,date,mack,text
0,2020-01-19,ACB,ACB có thể chọn đối tác độc quyền bancassuranc...
1,2020-01-22,ACB,ACB: Báo cáo quản trị công ty năm 2019
2,2020-02-10,ACB,ACB sẽ họp ĐHĐCĐ thường niên vào ngày 7/4
3,2020-02-10,ACB,ACB: Ngày ĐKCC để thực hiện quyền tham dự họp ...
4,2020-02-19,ACB,"ACB: 5.3.2020, ngày GDKHQ tham dự Đại hội đồng..."


In [7]:
@torch.no_grad()
def batch_phobert_sentiment(texts, max_length=256):
    """
    Input:
        texts: list[str]
    Output:
        probs: np.ndarray shape (batch, 3)
               [pos, neu, neg]
    """

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        outputs = model(**encodings)
    logits = outputs.logits                     # (batch, 3)
    probs = F.softmax(logits, dim=1)            # softmax

    return probs.cpu().numpy().astype(np.float32)


In [ ]:
# ==============================
# APPLY SENTIMENT (BATCH + CUDA + PROGRESS)
# ==============================

import numpy as np
from tqdm import tqdm

# ------------------------------
# CONFIG
# ------------------------------
BATCH_SIZE = 64 if device.type == "cuda" else 16
MAX_LEN = 256
TEXT_COL = "text"      # cột chứa nội dung news

texts = news[TEXT_COL].fillna("").astype(str).tolist()
n = len(texts)

all_probs = np.empty((n, 3), dtype=np.float32)

# ------------------------------
# BATCH INFERENCE
# ------------------------------
for i in tqdm(range(0, n, BATCH_SIZE), desc="PhoBERT Sentiment"):
    batch_texts = texts[i:i + BATCH_SIZE]

    probs = batch_phobert_sentiment(
        batch_texts,
        max_length=MAX_LEN
    )  # shape: (batch, 3)

    all_probs[i:i + len(batch_texts)] = probs

news["sent_pos"] = all_probs[:, 0]
news["sent_neu"] = all_probs[:, 1]
news["sent_neg"] = all_probs[:, 2]

# sentiment score (liên tục)
news["sent_score"] = news["sent_pos"] - news["sent_neg"]

# ------------------------------
# MAP TO {-1, 0, 1}
# ------------------------------
# argmax: 0=pos, 1=neu, 2=neg
sent_label_idx = all_probs.argmax(axis=1)

news["sentiment_label"] = np.select(
    [
        sent_label_idx == 0,
        sent_label_idx == 1,
        sent_label_idx == 2
    ],
    [
        1,   # positive
        0,   # neutral
       -1    # negative
    ]
)

news[[
    "sent_pos",
    "sent_neu",
    "sent_neg",
    "sent_score",
    "sentiment_label"
]].head()


PhoBERT Sentiment:  73%|███████▎  | 1625/2218 [15:42<05:32,  1.78it/s]

In [ ]:
sent_daily = (
    news.groupby(["mack","date"])
    .agg({
        "sent_pos":"mean",
        "sent_neu":"mean",
        "sent_neg":"mean",
        "sent_score":"mean"
    }).reset_index()
)


In [ ]:
# MERGE FULL DATASET
# ==============================

df = (
    ohlc
    .merge(fundamental_daily, on=["mack", "date"], how="left")
    .merge(sent_daily, on=["mack", "date"], how="left")
)

df = df.sort_values(["mack", "date"]).reset_index(drop=True)

print("Merged shape:", df.shape)
df.head()

Merged shape: (144176, 28)


,date,mack,open,high,low,close,volume,sma_10,sma_20,ema_10,...,roa,pb,pe,lnst_yoy,nophaitra_vcsh,vonhoa_tts,sent_pos,sent_neu,sent_neg,sent_score
0,2020-01-02,ACB,6580.0,6640.0,6560.0,6640.0,1163109,NaN,NaN,6640.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-03,ACB,6640.0,6690.0,6600.0,6640.0,1055528,NaN,NaN,6640.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-06,ACB,6640.0,6640.0,6500.0,6500.0,1286035,NaN,NaN,6614.545455,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-07,ACB,6500.0,6560.0,6500.0,6500.0,1050934,NaN,NaN,6593.719008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-08,ACB,6500.0,6500.0,6350.0,6380.0,2304937,NaN,NaN,6554.861007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ==============================
# FINAL NaN HANDLING (SAFE)
# ==============================

fund_cols = [
    "eps", "roe", "roa", "pb", "pe",
    "lnst_yoy", "nophaitra_vcsh", "vonhoa_tts"
]

sent_cols = ["sent_score", "sent_pos", "sent_neg", "sent_neu"]

# Sort to keep time order per ticker
df = df.sort_values(["mack", "date"]).reset_index(drop=True)

# Fundamentals: forward fill per ticker (no backfill to avoid leakage)
df[fund_cols] = (
    df.groupby("mack", group_keys=False)[fund_cols]
      .transform(lambda x: x.ffill())
)

# Sentiment: forward fill up to 5 days, then set remaining gaps to 0
existing_sent_cols = [c for c in sent_cols if c in df.columns]
df[existing_sent_cols] = (
    df.groupby("mack", group_keys=False)[existing_sent_cols]
      .transform(lambda x: x.ffill(limit=5))
)
df[existing_sent_cols] = df[existing_sent_cols].fillna(0.0)

# Drop rows missing required features
required_cols = fund_cols + existing_sent_cols
before = df.shape[0]
df = df.dropna(subset=required_cols).reset_index(drop=True)
after = df.shape[0]

print("NaN handling completed safely")
print(f"Rows before: {before:,}")
print(f"Rows after : {after:,}")
print(f"Dropped    : {before - after:,}")

In [ ]:
# 1. LIBRARIES
# =====================
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score


RANDOM_STATE = 42
sns.set_style("whitegrid")

In [ ]:
# ==============================
# SECTOR MAPPING (11 NGÀNH)
# ==============================

sector_map = {
    # 1. Dầu khí
    "HPG":"Dầu khí","HSG":"Dầu khí","NKG":"Dầu khí","PVT":"Dầu khí","PVD":"Dầu khí",

    # 2. Nguyên vật liệu
    "GVR":"Nguyên vật liệu","PHR":"Nguyên vật liệu","DGC":"Nguyên vật liệu",
    "DCM":"Nguyên vật liệu","DPM":"Nguyên vật liệu","BMP":"Nguyên vật liệu",
    "VGC":"Nguyên vật liệu","HT1":"Nguyên vật liệu",

    # 3. Công nghiệp
    "GEE":"Công nghiệp","GEX":"Công nghiệp","HDG":"Công nghiệp","PC1":"Công nghiệp",
    "REE":"Công nghiệp","MSN":"Công nghiệp","VIC":"Công nghiệp","GMD":"Công nghiệp",
    "VSC":"Công nghiệp","SCS":"Công nghiệp","VJC":"Công nghiệp","VCG":"Công nghiệp",
    "CII":"Công nghiệp","HHV":"Công nghiệp","PTB":"Công nghiệp","CTD":"Công nghiệp",
    "VTP":"Công nghiệp",

    # 4. Hàng tiêu dùng
    "TLG":"Hàng tiêu dùng","VNM":"Hàng tiêu dùng","HAG":"Hàng tiêu dùng",
    "SBT":"Hàng tiêu dùng","KDC":"Hàng tiêu dùng","VHC":"Hàng tiêu dùng",
    "DBC":"Hàng tiêu dùng","ANV":"Hàng tiêu dùng","PAN":"Hàng tiêu dùng",
    "SAB":"Hàng tiêu dùng","PNJ":"Hàng tiêu dùng",

    # 5. Dược phẩm & Y tế
    "IMP":"Dược phẩm và Y tế",

    # 6. Dịch vụ tiêu dùng
    "MWG":"Dịch vụ Tiêu dùng","FRT":"Dịch vụ Tiêu dùng","PLX":"Dịch vụ Tiêu dùng",
    "TCH":"Dịch vụ Tiêu dùng","DGW":"Dịch vụ Tiêu dùng",

    # 7. Viễn thông
    "CTR":"Viễn thông",

    # 8. Tiện ích cộng đồng
    "POW":"Tiện ích cộng đồng","NT2":"Tiện ích cộng đồng",
    "PPC":"Tiện ích cộng đồng","GAS":"Tiện ích cộng đồng","BWE":"Tiện ích cộng đồng",

    # 9. Tài chính
    "HDC":"Tài chính","VHM":"Tài chính","VRE":"Tài chính","BCM":"Tài chính",
    "KDH":"Tài chính","KBC":"Tài chính","DXG":"Tài chính","PDR":"Tài chính",
    "VPI":"Tài chính","SJS":"Tài chính","NLG":"Tài chính","DIG":"Tài chính",
    "SIP":"Tài chính","KOS":"Tài chính","DXS":"Tài chính","SZC":"Tài chính",
    "SSI":"Tài chính","VIX":"Tài chính","VND":"Tài chính","VCI":"Tài chính",
    "HCM":"Tài chính","FTS":"Tài chính","BSI":"Tài chính","DSE":"Tài chính",
    "CTS":"Tài chính","BVH":"Tài chính","EVF":"Tài chính",

    # 10. Ngân hàng
    "VCB":"Ngân hàng","CTG":"Ngân hàng","BID":"Ngân hàng","TCB":"Ngân hàng",
    "VPB":"Ngân hàng","MBB":"Ngân hàng","HDB":"Ngân hàng","LPB":"Ngân hàng",
    "ACB":"Ngân hàng","STB":"Ngân hàng","SHB":"Ngân hàng","VIB":"Ngân hàng",
    "SSB":"Ngân hàng","TPB":"Ngân hàng","EIB":"Ngân hàng","MSB":"Ngân hàng",
    "OCB":"Ngân hàng","NAB":"Ngân hàng",

    # 11. Công nghệ thông tin
    "FPT":"Công nghệ thông tin","CMG":"Công nghệ thông tin"
}
df["sector"] = df["mack"].map(sector_map)
df = df.dropna(subset=["sector"])

In [ ]:
# ======================================
# 2. LOAD & SORT DATA
# ======================================
# df must contain: mack, date, sector, close + features

df = df.copy()

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["mack", "date"]).reset_index(drop=True)

print("Initial shape:", df.shape)
print("Number of sectors:", df["sector"].nunique())
print("Sectors:", sorted(df["sector"].dropna().unique()))


Initial shape: (72464, 29)
Number of sectors: 11
Sectors: ['Công nghiệp', 'Công nghệ thông tin', 'Dược phẩm và Y tế', 'Dầu khí', 'Dịch vụ Tiêu dùng', 'Hàng tiêu dùng', 'Nguyên vật liệu', 'Ngân hàng', 'Tiện ích cộng đồng', 'Tài chính', 'Viễn thông']


# ======================================
# 4. TARGET: 1-DAY RETURN
# ======================================
df["return_1d"] = df.groupby("mack")["close"].pct_change().shift(-1)
df["target"] = np.where(
    df["return_1d"] > 0.005, 1,
    np.where(df["return_1d"] < -0.005, 0, np.nan)
)
df = df.dropna(subset=["target"]).reset_index(drop=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm import tqdm

# =====================
# 1. CREATE TARGET (PER TICKER)
# =====================
df = df.sort_values(["mack", "date"]).reset_index(drop=True)
next_close = df.groupby("mack")["close"].shift(-1)
df["target"] = (next_close > df["close"]).astype("Int64")
df = df.dropna(subset=["target"]).reset_index(drop=True)
df["target"] = df["target"].astype(int)

# =====================
# 2. FEATURE GROUPS
# =====================
fundamental_features = [
    "pe", "pb", "roe", "roa", "eps", "lnst_yoy",
    "vonhoa_tts", "nophaitra_vcsh"
]

technical_features = [
    "rsi_14", "macd", "macd_signal", "macd_hist",
    "sma_20", "sma_50", "ema_20",
    "volatility_20", "volume_ratio"
]

sentiment_features = [
    "sent_score", "sent_pos", "sent_neg"
]

feature_groups = {
    "Model 1 | Fundamental": fundamental_features,
    "Model 2 | Technical": technical_features,
    "Model 3 | Sentiment": sentiment_features,
    "Model 4 | Fund + Tech": fundamental_features + technical_features,
    "Model 5 | Fund + Sent": fundamental_features + sentiment_features,
    "Model 6 | Tech + Sent": technical_features + sentiment_features,
    "Model 7 | All": fundamental_features + technical_features + sentiment_features
}

# =====================
# 3. TRAIN FUNCTION
# =====================
def train_mlp(X_train, y_train, X_test, y_test):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=600,
            random_state=42
        ))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    }, y_pred, y_prob

# =====================
# 4. MAIN LOOP
# =====================
unique_dates = df["date"].sort_values().unique()
if len(unique_dates) < 2:
    raise ValueError("Not enough unique dates to split train/test.")
split_idx = int(len(unique_dates) * 0.8)
split_idx = max(1, min(len(unique_dates) - 1, split_idx))
split_date = pd.Timestamp(unique_dates[split_idx])
print("Split date:", split_date.date())

results = []
all_y_true = []
all_y_pred = []
all_y_prob = []
model_for_eval = "Model 7 | All"

for sector in tqdm(sorted(df["sector"].unique()), desc="Training sectors"):
    df_sector = df[df["sector"] == sector].copy()
    df_sector = df_sector.sort_values("date")

    if len(df_sector) < 150:
        continue

    for model_name, features in feature_groups.items():
        available_features = [f for f in features if f in df_sector.columns]
        if len(available_features) < 3:
            continue

        subset = df_sector[["date", "target"] + available_features].dropna()
        if len(subset) < 150:
            continue

        train_mask = subset["date"] < split_date
        if train_mask.sum() < 50 or (~train_mask).sum() < 50:
            continue

        X_train = subset.loc[train_mask, available_features]
        y_train = subset.loc[train_mask, "target"]
        X_test = subset.loc[~train_mask, available_features]
        y_test = subset.loc[~train_mask, "target"]

        if y_train.nunique() < 2 or y_test.nunique() < 2:
            continue

        metrics, y_pred, y_prob = train_mlp(X_train, y_train, X_test, y_test)

        results.append({
            "Sector": sector,
            "Model": model_name,
            "Accuracy": metrics["Accuracy"],
            "AUC": metrics["AUC"],
            "n_obs": len(subset)
        })

        if model_name == model_for_eval:
            all_y_true.append(y_test.values)
            all_y_pred.append(y_pred)
            all_y_prob.append(y_prob)

results_df = pd.DataFrame(results)
print("Finished training")
print("Number of sectors in result:", results_df["Sector"].nunique())

auc_table = results_df.pivot(
    index="Sector",
    columns="Model",
    values="AUC"
).round(3)

acc_table = results_df.pivot(
    index="Sector",
    columns="Model",
    values="Accuracy"
).round(3)

In [ ]:
# =====================
# 5. HEATMAPS (AUC / ACCURACY)
# =====================
plt.figure(figsize=(12, 6))
sns.heatmap(auc_table, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("AUC by Sector and Feature Group (MLP)")
plt.xlabel("Feature Group")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
sns.heatmap(acc_table, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Accuracy by Sector and Feature Group (MLP)")
plt.xlabel("Feature Group")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

In [ ]:
# =====================
# 6. SUMMARY BY MODEL
# =====================
summary = (
    results_df
    .groupby("Model")[["AUC", "Accuracy"]]
    .agg(["mean", "median", "std"])
    .round(3)
)
summary

summary_reset = summary.reset_index()
summary_reset.columns = [
    "Model",
    "AUC_mean", "AUC_median", "AUC_std",
    "Accuracy_mean", "Accuracy_median", "Accuracy_std"
]

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_reset, x="Model", y="AUC_mean", color="#4C78A8")
plt.title("Mean AUC by Feature Group")
plt.xlabel("Feature Group")
plt.ylabel("Mean AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_reset, x="Model", y="Accuracy_mean", color="#F58518")
plt.title("Mean Accuracy by Feature Group")
plt.xlabel("Feature Group")
plt.ylabel("Mean Accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# =====================
# 7. METRIC DISTRIBUTIONS
# =====================
plt.figure(figsize=(10, 5))
sns.boxplot(data=results_df, x="Model", y="AUC")
plt.title("AUC Distribution by Feature Group")
plt.xlabel("Feature Group")
plt.ylabel("AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=results_df, x="Model", y="Accuracy")
plt.title("Accuracy Distribution by Feature Group")
plt.xlabel("Feature Group")
plt.ylabel("Accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# =====================
# 8. CONFUSION MATRIX & ROC (MODEL 7 | ALL)
# =====================
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

if len(all_y_true) > 0:
    y_true = np.concatenate(all_y_true)
    y_prob = np.concatenate(all_y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    print(classification_report(y_true, y_pred, digits=3))

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax[0], colorbar=False)
    ax[0].set_title("Confusion Matrix (Model 7 | All)")

    RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax[1])
    ax[1].set_title("ROC Curve (Model 7 | All)")

    plt.tight_layout()
    plt.show()
else:
    print("No aggregated predictions available for Model 7 | All.")

In [ ]:
# =====================
# 9. DATA DISTRIBUTION
# =====================
sector_counts = df["sector"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=sector_counts.values, y=sector_counts.index, color="#54A24B")
plt.title("Observations by Sector")
plt.xlabel("Number of rows")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

target_counts = df["target"].value_counts(normalize=True).sort_index()

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, color="#E45756")
plt.title("Target Balance (Up=1, Down=0)")
plt.xlabel("Target")
plt.ylabel("Share")
plt.tight_layout()
plt.show()

In [ ]:
# =====================
# 10. SAMPLE STOCK + FEATURE CORRELATION + SENTIMENT
# =====================
sample_mack = df["mack"].value_counts().idxmax()
sample_df = df[df["mack"] == sample_mack].sort_values("date").set_index("date")

plt.figure(figsize=(12, 5))
plt.plot(sample_df.index, sample_df["close"], label="Close", linewidth=1.5)
plt.plot(sample_df.index, sample_df["sma_20"], label="SMA 20", alpha=0.8)
plt.plot(sample_df.index, sample_df["sma_50"], label="SMA 50", alpha=0.8)
plt.plot(sample_df.index, sample_df["ema_20"], label="EMA 20", alpha=0.8)
plt.title(f"Price and Indicators for {sample_mack}")
plt.legend()
plt.tight_layout()
plt.show()

corr_cols = [c for c in (fundamental_features + technical_features + sentiment_features) if c in df.columns]
corr_data = df[corr_cols].dropna()
if len(corr_data) > 5000:
    corr_data = corr_data.sample(5000, random_state=42)

plt.figure(figsize=(12, 8))
sns.heatmap(corr_data.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation (Sampled)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["sent_score"], bins=50, kde=True, color="#B279A2")
plt.title("Sentiment Score Distribution")
plt.xlabel("sent_score")
plt.tight_layout()
plt.show()